In [2]:
install.packages("xgboost")
install.packages("Matrix")
install.packages("ggplot2")
install.packages("lattice")
install.packages("caret")
install.packages("dplyr")
install.packages("tictoc")

Installation du package dans ‘/home/pacome/R/x86_64-pc-linux-gnu-library/4.2’
(car ‘lib’ n'est pas spécifié)

Installation du package dans ‘/home/pacome/R/x86_64-pc-linux-gnu-library/4.2’
(car ‘lib’ n'est pas spécifié)

Warning message in install.packages("Matrix"):
“l'installation du package ‘Matrix’ a eu un statut de sortie non nul”
Installation du package dans ‘/home/pacome/R/x86_64-pc-linux-gnu-library/4.2’
(car ‘lib’ n'est pas spécifié)

Installation du package dans ‘/home/pacome/R/x86_64-pc-linux-gnu-library/4.2’
(car ‘lib’ n'est pas spécifié)

Installation du package dans ‘/home/pacome/R/x86_64-pc-linux-gnu-library/4.2’
(car ‘lib’ n'est pas spécifié)

installation des dépendances ‘proxy’, ‘e1071’, ‘ModelMetrics’, ‘pROC’, ‘reshape2’


Installation du package dans ‘/home/pacome/R/x86_64-pc-linux-gnu-library/4.2’
(car ‘lib’ n'est pas spécifié)

Installation du package dans ‘/home/pacome/R/x86_64-pc-linux-gnu-library/4.2’
(car ‘lib’ n'est pas spécifié)



In [3]:
library(Matrix)
library(xgboost)
library(ggplot2)
library(lattice)
library(caret)
library(dplyr)
library(tictoc)


Attachement du package : ‘dplyr’


L'objet suivant est masqué depuis ‘package:xgboost’:

    slice


Les objets suivants sont masqués depuis ‘package:stats’:

    filter, lag


Les objets suivants sont masqués depuis ‘package:base’:

    intersect, setdiff, setequal, union




In [6]:
data <- read.csv("data_target_encoding.csv",stringsAsFactors = T)

In [5]:
options(repr.matrix.max.cols=100)
data[1:3,]

,geo_level_1_mean_damage,geo_level_1_sd_damage,geo_level_2_mean_damage,geo_level_2_sd_damage,geo_level_3_mean_damage,geo_level_3_sd_damage,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,foundation_type,roof_type,ground_floor_type,other_floor_type,position,plan_configuration,has_superstructure_adobe_mud,has_superstructure_mud_mortar_stone,has_superstructure_stone_flag,has_superstructure_cement_mortar_stone,has_superstructure_mud_mortar_brick,has_superstructure_cement_mortar_brick,has_superstructure_timber,has_superstructure_bamboo,has_superstructure_rc_non_engineered,has_superstructure_rc_engineered,has_superstructure_other,legal_ownership_status,count_families,has_secondary_use_agriculture,has_secondary_use_hotel,has_secondary_use_rental,has_secondary_use_institution,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other,damage_grade
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<fct>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,2.020477,0.4558226,1.984293,0.2987067,1.971429,0.1690309,1,25,5,2,t,r,n,f,j,s,d,0,1,0,0,0,0,0,0,0,0,0,v,0,0,0,0,0,0,0,0,0,0,0,2
2,2.794480,0.4352255,2.977444,0.1490457,3.000000,0.0000000,2,0,13,7,t,r,n,f,q,s,d,0,1,0,0,0,0,0,0,0,0,0,v,1,0,0,0,0,0,0,0,0,0,0,3
3,2.794480,0.4352255,2.984772,0.1227723,3.000000,0.0000000,2,5,12,6,o,r,q,f,q,s,d,0,1,0,0,0,0,0,0,0,0,0,v,1,0,0,0,0,0,0,0,0,0,0,3


In [10]:
X <- sparse.model.matrix(damage_grade ~ .,data = data)[,-1]
y <- as.numeric(data[,c("damage_grade")]-1)

Xgboost 5cv sans tuning 

In [12]:
k = 5
accuracy_vec <- array(0,k)

spam_idx <- sample(1:nrow(X))
max <- ceiling(nrow(X)/k)
splits <- split(spam_idx, ceiling(seq_along(spam_idx)/max)) # permet de gérer le fait qu'on split en des group de tail differente

tic()
pb2 <- txtProgressBar(min = 1, max = k, style = 3)

for (i in 1:k){
    
    X.test <- X[splits[[i]],]
    y.test <- y[splits[[i]]]
    X.train <- X[-splits[[i]],]
    y.train <- y[-splits[[i]]]
    
    grid_default <- expand.grid(
    nrounds = 100,
    max_depth = 6,
    eta = 0.3,
    gamma = 0,
    colsample_bytree = 1,
    min_child_weight = 1,
    subsample = 1
    )

    train_control <- caret::trainControl(
    method = "none",
    verboseIter = FALSE, # no training log
    allowParallel = TRUE, # FALSE for reproducible results 
    summaryFunction = multiClassSummary
    )

    xgb_model <- caret::train(
    x = X.train,
    y = as.factor(y.train),
    trControl = train_control,
    tuneGrid = grid_default,
    method = "xgbTree",
    verbose = TRUE
    )

    pred <- predict(xgb_model, X.test)
    ll <- length(y.test)
    cfm <- matrix(0, nrow = 3, ncol = 3)
    for (j in 1:ll){cfm[pred[j],(y.test[j]+1)] <- cfm[pred[j],(y.test[j]+1)]+1}
    TP <- c(cfm[1,1],cfm[2,2],cfm[3,3])
    FP <- c(cfm[2,1]+cfm[3,1],cfm[1,2]+cfm[3,2],cfm[1,3]+cfm[2,3])
    FN <- c(cfm[1,2]+cfm[1,3],cfm[2,1]+cfm[2,3],cfm[3,1]+cfm[3,2])
    P_micro <- sum(TP)/sum(TP+FP)
    R_micro <- sum(TP)/sum(TP+FN)
    
    accuracy_vec[i] <- (2*P_micro*R_micro)/(P_micro+R_micro)
    
    setTxtProgressBar(pb2, i)
    print((2*P_micro*R_micro)/(P_micro+R_micro))   
}

mean(accuracy_vec)
toc()

  |                                                                      |   0%[1] 0.7621496
  |==================                                                    |  25%[1] 0.760538
  |===================================                                   |  50%[1] 0.7577944
  |====================================================                  |  75%[1] 0.7642985
  |======================================================================| 100%[1] 0.7623616


[1] 0.7614284

299.141 sec elapsed


xgboost 5cv subsample

In [16]:
k = 5
accuracy_vec.sub <- array(0,k)


spam_idx <- sample(1:nrow(data))
max <- ceiling(nrow(data)/k)
splits <- split(spam_idx, ceiling(seq_along(spam_idx)/max)) # permet de gérer le fait qu'on split en des group de tail differente


tic()
pb2 <- txtProgressBar(min = 1, max = k, style = 3)

for (i in 1:k){
    
    data.test <- data[splits[[i]],]
    data.train <- data[-splits[[i]],]
    
    idx_damage_grade_1 <- which(data.train$damage_grade == 1)
    n_sub <- length(idx_damage_grade_1)

    idx_damage_grade_2 <- sample(which(data.train$damage_grade == 2),n_sub)
    idx_damage_grade_3 <- sample(which(data.train$damage_grade == 3),n_sub)
    
    data.sub.train <- rbind(data.train[idx_damage_grade_1,],data.train[idx_damage_grade_2,],data.train[idx_damage_grade_3,])
    

    X.sub.train <- sparse.model.matrix(damage_grade ~ .,data = data.sub.train)[,-1]
    X.test <- sparse.model.matrix(damage_grade ~ .,data = data.test)[,-1]
    y.sub.train <- as.numeric(data.sub.train[,c("damage_grade")]-1)
    y.test <- as.numeric(data.train[,c("damage_grade")]-1)
    
    
    grid_default <- expand.grid(
    nrounds = 100,
    max_depth = 6,
    eta = 0.3,
    gamma = 0,
    colsample_bytree = 1,
    min_child_weight = 1,
    subsample = 1
    )

    train_control <- caret::trainControl(
    method = "none",
    verboseIter = FALSE, # no training log
    allowParallel = TRUE, # FALSE for reproducible results 
    summaryFunction = multiClassSummary
    )

    xgb_model <- caret::train(
    x = X.sub.train,
    y = as.factor(y.sub.train),
    trControl = train_control,
    tuneGrid = grid_default,
    method = "xgbTree",
    verbose = TRUE
    )

    pred <- predict(xgb_model, X.test)
    ll <- length(y.test)
    cfm <- matrix(0, nrow = 3, ncol = 3)
    for (j in 1:ll){cfm[pred[j],(y.test[j]+1)] <- cfm[pred[j],(y.test[j]+1)]+1}
    TP <- c(cfm[1,1],cfm[2,2],cfm[3,3])
    FP <- c(cfm[2,1]+cfm[3,1],cfm[1,2]+cfm[3,2],cfm[1,3]+cfm[2,3])
    FN <- c(cfm[1,2]+cfm[1,3],cfm[2,1]+cfm[2,3],cfm[3,1]+cfm[3,2])
    P_micro <- sum(TP)/sum(TP+FP)
    R_micro <- sum(TP)/sum(TP+FN)
    
    accuracy_vec.sub[i] <- (2*P_micro*R_micro)/(P_micro+R_micro)
    
    setTxtProgressBar(pb2, i)
    print((2*P_micro*R_micro)/(P_micro+R_micro))   
}

mean(accuracy_vec.sub)
toc()

  |                                                                      |   0%[1] 0.39109
  |==================                                                    |  25%[1] 0.3944667
  |===================================                                   |  50%[1] 0.391915
  |====================================================                  |  75%[1] 0.3950807
  |======================================================================| 100%[1] 0.3947272


[1] 0.3934559

109.546 sec elapsed


xgboost 5cv oversample

In [19]:
k = 5
accuracy_vec.over <- array(0,k)


spam_idx <- sample(1:nrow(data))
max <- ceiling(nrow(data)/k)
splits <- split(spam_idx, ceiling(seq_along(spam_idx)/max)) # permet de gérer le fait qu'on split en des group de tail differente


tic()
pb2 <- txtProgressBar(min = 1, max = k, style = 3)

for (i in 1:k){
    
    data.test <- data[splits[[i]],]
    data.train <- data[-splits[[i]],]
    
    idx_damage_grade_1 <- which(data.train$damage_grade == 1)
    idx_damage_grade_2 <- which(data.train$damage_grade == 2)
    idx_damage_grade_3 <- which(data.train$damage_grade == 3)
    
    n_over1 <- length(idx_damage_grade_2)-length(idx_damage_grade_1)
    n_over3 <- length(idx_damage_grade_2)-length(idx_damage_grade_1)
    
    idx_damage_grade_1 <- sample(idx_damage_grade_1,n_over1,replace=TRUE)
    idx_damage_grade_3 <- sample(idx_damage_grade_3,n_over3,replace=TRUE)
    
    data.over.train <- rbind(data.train[idx_damage_grade_1,],data.train[idx_damage_grade_2,],data.train[idx_damage_grade_3,])
    

    X.over.train <- sparse.model.matrix(damage_grade ~ .,data = data.over.train)[,-1]
    X.test <- sparse.model.matrix(damage_grade ~ .,data = data.test)[,-1]
    y.over.train <- as.numeric(data.over.train[,c("damage_grade")]-1)
    y.test <- as.numeric(data.train[,c("damage_grade")]-1)
    
    
    grid_default <- expand.grid(
    nrounds = 100,
    max_depth = 6,
    eta = 0.3,
    gamma = 0,
    colsample_bytree = 1,
    min_child_weight = 1,
    subsample = 1
    )

    train_control <- caret::trainControl(
    method = "none",
    verboseIter = FALSE, # no training log
    allowParallel = TRUE, # FALSE for reproducible results 
    summaryFunction = multiClassSummary
    )

    xgb_model <- caret::train(
    x = X.over.train,
    y = as.factor(y.over.train),
    trControl = train_control,
    tuneGrid = grid_default,
    method = "xgbTree",
    verbose = TRUE
    )

    pred <- predict(xgb_model, X.test)
    ll <- length(y.test)
    cfm <- matrix(0, nrow = 3, ncol = 3)
    for (j in 1:ll){cfm[pred[j],(y.test[j]+1)] <- cfm[pred[j],(y.test[j]+1)]+1}
    TP <- c(cfm[1,1],cfm[2,2],cfm[3,3])
    FP <- c(cfm[2,1]+cfm[3,1],cfm[1,2]+cfm[3,2],cfm[1,3]+cfm[2,3])
    FN <- c(cfm[1,2]+cfm[1,3],cfm[2,1]+cfm[2,3],cfm[3,1]+cfm[3,2])
    P_micro <- sum(TP)/sum(TP+FP)
    R_micro <- sum(TP)/sum(TP+FN)
    
    accuracy_vec.over[i] <- (2*P_micro*R_micro)/(P_micro+R_micro)
    
    setTxtProgressBar(pb2, i)
    print((2*P_micro*R_micro)/(P_micro+R_micro))   
}

mean(accuracy_vec.over)
toc()

  |                                                                      |   0%[1] 0.413154
  |==================                                                    |  25%[1] 0.4112738
  |===================================                                   |  50%[1] 0.4107174
  |====================================================                  |  75%[1] 0.4078011
  |======================================================================| 100%[1] 0.4090604


[1] 0.4104013

447.217 sec elapsed
